In [1]:
from getpass import getuser # Libaray to copy things
from pathlib import Path # Object oriented libary to deal with paths
from dask.utils import format_bytes
from distributed import Client, progress, wait # Libaray to orchestrate distributed resources
#from dask_jobqueue import SLURMCluster # Setting up distributed memories via slurm
import numpy as np # Pythons standard array library
import xarray as xr # Libary to work with labeled n-dimensional data
import dask # Distributed data libary
import dask.distributed
import multiprocessing
from dask.diagnostics import ProgressBar
import glob, os
from matplotlib.lines import Line2D
from scipy import stats

from esda.moran import Moran
from libpysal.weights import KNN
from matplotlib.patches import Patch
from scipy.ndimage import gaussian_filter1d
from subprocess import run, PIPE
import sys
import os
from sklearn.neighbors import KernelDensity
import gc
from concurrent.futures import ProcessPoolExecutor
import seaborn as sns
import xskillscore as xs
import warnings
warnings.filterwarnings(action='ignore')
dask.config.config.get('distributed').get('dashboard').update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})
import pandas as pd
import re

from collections import defaultdict
from scipy.interpolate import CloughTocher2DInterpolator, LinearNDInterpolator, NearestNDInterpolator

In [2]:
#'''Provides information about the CPU and starts the dask client'''
ncpu = multiprocessing.cpu_count()
processes = False
nworker = 4
threads = ncpu // nworker
print(f"Number of CPUs: {ncpu}, number of threads: {threads}, number of workers: {nworker}, processes: {processes}")
client = Client(processes=processes,
 threads_per_worker=threads,
 n_workers=nworker,
 memory_limit='1000GB'
 )

Number of CPUs: 64, number of threads: 16, number of workers: 4, processes: False


2026-02-16 15:50:48,562 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)
2026-02-16 15:50:55,915 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)
2026-02-16 15:51:00,421 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)
2026-02-16 15:51:11,420 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)
2026-02-16 15:51:20,798 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)
2026-02-16 15:52:18,568 - distributed.utils_perf - WARNING - full garbage collections took 11% CPU time recently (threshold: 10%)
2026-02-16 15:55:13,335 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)
2026-02-16 15:57:42,221 - distributed.utils_perf - WARNING - full garbage collections took

In [3]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jwille//proxy/39241/status,
Dashboard: /user/jwille//proxy/39241/status,Workers: 4
Total threads: 64,Total memory: 3.64 TiB
Status: running,Using processes: False
Comm: inproc://129.132.32.66/55431/1,Workers: 4
Dashboard: /user/jwille//proxy/39241/status,Total threads: 64
Started: Just now,Total memory: 3.64 TiB
Comm: inproc://129.132.32.66/55431/4,Total threads: 16
Dashboard: /user/jwille//proxy/39309/status,Memory: 0.91 TiB
Nanny: None,


In [4]:

def markov_transitions_wetdry(x):
    """
    x: DataArray with values {0,1} and possibly NaNs (e.g., ocean mask).
       Here: 1=wet, 0=dry.
    Returns: Dataset with P_WW, P_WD, P_DW, P_DD and counts.
    """
    # t and t+1 on the SAME time coordinate (positional pairing)
    x0 = x.isel(time=slice(None, -1))
    x1 = x.shift(time=-1).isel(time=slice(None, -1))  # aligns to x0.time

    # Valid where neither step is NaN
    valid = x0.notnull() & x1.notnull()

    # Boolean masks for states
    W0 = (x0 == 1) & valid
    D0 = (x0 == 0) & valid
    W1 = (x1 == 1) & valid
    D1 = (x1 == 0) & valid

    # Transition counts
    n_WW = (W0 & W1).sum('time')
    n_WD = (W0 & D1).sum('time')
    n_DW = (D0 & W1).sum('time')
    n_DD = (D0 & D1).sum('time')

    # Denominators: number of times we were in each state at t
    n_W = W0.sum('time')
    n_D = D0.sum('time')

    # Probabilities (avoid divide-by-zero -> NaN where undefined)
    P_WW = (n_WW / n_W).where(n_W > 0)
    P_WD = (n_WD / n_W).where(n_W > 0)
    P_DW = (n_DW / n_D).where(n_D > 0)
    P_DD = (n_DD / n_D).where(n_D > 0)

    return xr.Dataset(
        {
            "P_WW": P_WW,
            "P_WD": P_WD,
            "P_DW": P_DW,
            "P_DD": P_DD,
            "n_W": n_W,
            "n_D": n_D,
            "n_WW": n_WW,
            "n_WD": n_WD,
            "n_DW": n_DW,
            "n_DD": n_DD,
        }
    )


## IFS deep on

In [ ]:
#IFS deep on land
latitude = [30,60]
model = "IFS"

data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_2.8km_deep_on/daily_precip/mask_1')
files = sorted([str(f) for f in data_path.glob('mask_1_IFS_2.8-FESOM_5_daily_precip_*.nc')])[:]
ifs_on = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['dry']
indexer = ((ifs_on.lat>=latitude[0]) & (ifs_on.lat<=latitude[1])).compute()
ifs_on = ifs_on.where(indexer,drop=True)
print('hello')
print(ifs_on)
out = markov_transitions_wetdry(ifs_on)
out = out.compute()

out.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/ifs_2.8km_on_30-60N_markov.nc')






## IFS deep off

In [ ]:
latitude = [30,60]
model = "IFS"

data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_2.8km_deep_off/daily_precip/mask_1')
files = sorted([str(f) for f in data_path.glob('mask_1_IFS_2.8-FESOM_5_deep_off_daily_precip_*.nc')])[:]
ifs_off = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['dry']
indexer = ((ifs_off.lat>=latitude[0]) & (ifs_off.lat<=latitude[1])).compute()
ifs_off = ifs_off.where(indexer,drop=True)
out = markov_transitions_wetdry(ifs_off)
out = out.compute()

out.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/ifs_2.8km_off_30-60N_markov.nc')


## IFS cycle3

In [ ]:
model = "IFS"
years = range(2020, 2025)

results_temporal = []

for year in years:
    print(f"Processing year {year}...")
    ds = xr.open_dataset(f"/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_4.4-FESOM_5/daily_precip/mask_1/mask_1_IFS_4.4-FESOM_5_precip_30-60N_{year}.nc")    
    dry_da = ds["dry"]
    out = markov_transitions_wetdry(dry_da)
    out = out.compute()
    out = out.expand_dims(dim={"year": [year]})

    results_temporal.append(out)

ifs_cycle3_markov = xr.concat(results_temporal, dim="year")
ifs_cycle3_markov.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/ifs_cycle3_30-60N_markov.nc')



## IFS cycle4

In [ ]:
model = "IFS"
years = range(1990, 2020)
#years = range(2020, 2050)

months = [5,6,7,8,9,10]

results_temporal = []

for year in years:
    print(f"Processing year {year}...")
    ds = xr.open_dataset(f"/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_zoom9/1990-2020/daily_precip/wet_30-60N_merged_{year}.nc")
    #ds = xr.open_dataset(f"/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_zoom9/2020-2050/daily_precip/wet_30-60N_merged_{year}.nc")
    ds = ds.where(ds.time.dt.month.isin(months), drop=True)
    dry_da = ds["dry"]
    out = markov_transitions_wetdry(dry_da)
    out = out.compute()
    out = out.expand_dims(dim={"year": [year]})

    results_temporal.append(out)

ifs_cycle4_markov = xr.concat(results_temporal, dim="year")
ifs_cycle4_markov.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/ifs_cycle4_1990-2020_30-60N_markov.nc')





## ICON cycle3

In [ ]:
model = "ICON"
years = range(2020, 2025)

results_temporal = []

for year in years:
    print(f"Processing year {year}...")
    ds = xr.open_dataset(f"/home/jwille/xenon_data/NextGEMS/NextGEMS/ICON_zoom10/daily_precip/mask_1/mask_1_ICON_ngc3028_hourly_precip_30-60N_{year}.nc")    
    dry_da = ds["dry"]
    out = markov_transitions_wetdry(dry_da)
    out = out.compute()
    out = out.expand_dims(dim={"year": [year]})

    results_temporal.append(out)

icon_cycle3_markov = xr.concat(results_temporal, dim="year")
icon_cycle3_markov.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/icon_cycle3_30-60N_markov.nc')



## ICON cycle4

In [ ]:
model = "ICON"
#years = range(2020, 2050)
months = [5,6,7,8,9,10]
years = range(2020, 2050)
results_temporal = []

for year in years:
    print(f"Processing year {year}...")
    ds = xr.open_dataset(f"/home/jwille/xenon_data/NextGEMS/NextGEMS/ICON_zoom9/2020-2050/daily_precip/wet_30-60N_merged_{year}.nc")
    ds = ds.where(ds.time.dt.month.isin(months), drop=True)
    dry_da = ds["dry"]
    out = markov_transitions_wetdry(dry_da)
    out = out.compute()
    out = out.expand_dims(dim={"year": [year]})

    results_temporal.append(out)

icon_cycle4_markov = xr.concat(results_temporal, dim="year")
icon_cycle4_markov.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/icon_cycle4_30-60N_markov.nc')


## IMERG

In [ ]:
model = "IMERG"

years = range(2014, 2024)

results_temporal = []

for year in years:
    print(f"Processing year {year}...")
    ds = xr.open_dataset(f"/home/jwille/xenon_data/NextGEMS/IMERG/clean/global/QC/daysum/mask_1/mask_1_test_30-60N_daysum_QC_IMERG_V07B_final_{year}.nc4")  
    dry_da = ds["dry"]
    out = markov_transitions_wetdry(dry_da)
    out = out.compute()
    out = out.expand_dims(dim={"year": [year]})

    results_temporal.append(out)

imerg_markov = xr.concat(results_temporal, dim="year")
imerg_markov.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/imerg_30-60N_markov.nc')



## CESM2

In [5]:
# Change this to your directory path
model = 'CESM2'
#Source of CESM2 data /net/atmos/data/CESM2-LE/day/PRECT
directory = '/home/jwille/xenon_data/NextGEMS/CESM2/precip_past'
#directory = '/home/jwille/xenon_data/NextGEMS/CESM2/precip_future'

# List all file names in the directory (adjust the filter as needed, e.g., for .nc files)
files = [f for f in os.listdir(directory) if f.endswith('.nc')]

# Create a dictionary to group files by simulation number
sim_files = defaultdict(list)


pattern = re.compile(r'LE2-1251\.(\d+)\.cam')

for file in files:
    match = pattern.search(file)
    if match:
        sim_num = match.group(1)
        sim_files[sim_num].append(file)
    else:
        print("Filename did not match expected pattern:", file)
# Now loop over each simulation number group and process the files for that simulation.
for sim, file_list in sim_files.items():
    print(f"Processing simulation {sim} with {len(file_list)} files:")
    # Here you can order or sort the files if the year range matters,
    # e.g., sorting by the start date in the filename (assuming YYYYMMDD is at a fixed position).
    # For example:
    file_list = sorted(file_list)

Processing simulation 014 with 17 files:
Processing simulation 015 with 17 files:
Processing simulation 016 with 17 files:
Processing simulation 017 with 17 files:
Processing simulation 018 with 17 files:
Processing simulation 019 with 17 files:
Processing simulation 020 with 17 files:


In [6]:
# Now loop over each simulation number group and process the files for that simulation.
for sim, file_list in sim_files.items():
    print(f"Processing simulation {sim} with {len(file_list)} files:")
    latitude = [30,60]
    months = [5,6,7,8,9,10]
    results_temporal = []
    
    file_list = sorted(file_list)
    # Create a full file path list
    full_paths = [os.path.join(directory, f) for f in file_list]

    #print(full_paths)
    dataset = xr.open_mfdataset(full_paths, engine='netcdf4',combine='by_coords')['PRECT']
    dataset = dataset * 86400 * 1000
    dataset = dataset.sel(time=slice('1850', '2015'))  # select time range 
    #dataset = dataset.sel(time=slice('2014', '2050'))  # select time range 
    dataset = dataset.where(dataset.time.dt.month.isin(months), drop=True)

    indexer = ((dataset.lat>=latitude[0]) & (dataset.lat<=latitude[1]))
    dataset = dataset.where(indexer,drop=True)
    dataset = dataset.convert_calendar('noleap')
    dataset = dataset.load()

    start_date = pd.Timestamp(''+str(dataset.time.dt.year.values[0]-1)+'-05-01')
    end_date = pd.Timestamp(''+str(dataset.time.dt.year.values[-1])+'-10-31')      
    year_range = dataset.time.dt.year.values[-1] - dataset.time.dt.year.values[0] + 1

    periods = pd.date_range(start_date, end_date, freq='Y')
    num_periods = len(periods)

    time_size = []
#    for i in range(num_periods - 1):
#        # Define the start and end dates for the current period
#        period_start = periods[i]
#        period_end = periods[i + 1]
#        dataset_sel = dataset.sel(time=slice(str(period_start),str(period_end)))
#        if len(dataset_sel.time) > 1:
#            time_size.append(len(dataset_sel.time))
#    print(dataset_sel.time)
#    result = np.zeros([len(time_size),len(dataset.lat),len(dataset.lon)])   


    
    for i in range(num_periods - 1):
        period_start = periods[i]
        period_end = periods[i + 1]
        print(period_start)
        print(period_end)
        dataset_sel = dataset.sel(time=slice(str(period_start),str(period_end)))
        if len(dataset_sel.time) > 1:
            dataset_sel = xr.where(dataset_sel > 1, 1.0, 0.0) 
            out = markov_transitions_wetdry(dataset_sel)
            out = out.compute()
            out = out.expand_dims(dim={"year": [dataset_sel.time.values[0].year]})
            results_temporal.append(out)

    cesm2_markov = xr.concat(results_temporal, dim="year")
    print(cesm2_markov)
    cesm2_markov = cesm2_markov.load()

    cesm2_markov.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/cesm2_1850-2014_'+sim+'_30-60N_markov.nc')
    
    #cesm2_temporal_corr_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/cesm2_2014-2049_'+sim+'_30-60N_temporal_corr.nc')




Processing simulation 014 with 17 files:
1849-12-31 00:00:00
1850-12-31 00:00:00
1850-12-31 00:00:00
1851-12-31 00:00:00
1851-12-31 00:00:00
1852-12-31 00:00:00
1852-12-31 00:00:00
1853-12-31 00:00:00
1853-12-31 00:00:00
1854-12-31 00:00:00
1854-12-31 00:00:00
1855-12-31 00:00:00
1855-12-31 00:00:00
1856-12-31 00:00:00
1856-12-31 00:00:00
1857-12-31 00:00:00
1857-12-31 00:00:00
1858-12-31 00:00:00
1858-12-31 00:00:00
1859-12-31 00:00:00
1859-12-31 00:00:00
1860-12-31 00:00:00
1860-12-31 00:00:00
1861-12-31 00:00:00
1861-12-31 00:00:00
1862-12-31 00:00:00
1862-12-31 00:00:00
1863-12-31 00:00:00
1863-12-31 00:00:00
1864-12-31 00:00:00
1864-12-31 00:00:00
1865-12-31 00:00:00
1865-12-31 00:00:00
1866-12-31 00:00:00
1866-12-31 00:00:00
1867-12-31 00:00:00
1867-12-31 00:00:00
1868-12-31 00:00:00
1868-12-31 00:00:00
1869-12-31 00:00:00
1869-12-31 00:00:00
1870-12-31 00:00:00
1870-12-31 00:00:00
1871-12-31 00:00:00
1871-12-31 00:00:00
1872-12-31 00:00:00
1872-12-31 00:00:00
1873-12-31 00:00:00

2026-02-16 15:50:44,257 - distributed.utils_perf - WARNING - full garbage collections took 10% CPU time recently (threshold: 10%)


1849-12-31 00:00:00
1850-12-31 00:00:00
1850-12-31 00:00:00
1851-12-31 00:00:00
1851-12-31 00:00:00
1852-12-31 00:00:00
1852-12-31 00:00:00
1853-12-31 00:00:00
1853-12-31 00:00:00
1854-12-31 00:00:00
1854-12-31 00:00:00
1855-12-31 00:00:00
1855-12-31 00:00:00
1856-12-31 00:00:00
1856-12-31 00:00:00
1857-12-31 00:00:00
1857-12-31 00:00:00
1858-12-31 00:00:00
1858-12-31 00:00:00
1859-12-31 00:00:00
1859-12-31 00:00:00
1860-12-31 00:00:00
1860-12-31 00:00:00
1861-12-31 00:00:00
1861-12-31 00:00:00
1862-12-31 00:00:00
1862-12-31 00:00:00
1863-12-31 00:00:00
1863-12-31 00:00:00
1864-12-31 00:00:00
1864-12-31 00:00:00
1865-12-31 00:00:00
1865-12-31 00:00:00
1866-12-31 00:00:00
1866-12-31 00:00:00
1867-12-31 00:00:00
1867-12-31 00:00:00
1868-12-31 00:00:00
1868-12-31 00:00:00
1869-12-31 00:00:00
1869-12-31 00:00:00
1870-12-31 00:00:00
1870-12-31 00:00:00
1871-12-31 00:00:00
1871-12-31 00:00:00
1872-12-31 00:00:00
1872-12-31 00:00:00
1873-12-31 00:00:00
1873-12-31 00:00:00
1874-12-31 00:00:00
